# OpenShift AI MaaS Demo: Intelligent News Triage & RAG Dispatcher

Tento notebook demonstruje enterprise využití OpenShift AI MaaS (Model-as-a-Service) pro automatizovanou triáž zpravodajství:

* **MaaS Gateway Integration:** Jednotné API rozhraní chráněné autentizací a řízením přístupu.
* **Guided Decoding (Structured JSON Output):** Vynucení přesné struktury výstupu bez rizika halucinací formátu.
* **Metering & Cost Tracking:** Sledování spotřeby tokenů a latence pro audity a rozpočítávání nákladů.
* **Dynamic RAG Ingestion:** Automatické směrování strukturovaných dat do navazujících RAG databází.

In [ ]:
import os
import json

# Vytvoření adresáře data/
os.makedirs("data", exist_ok=True)

# Vzorkové zpravodajské texty z různých oblastí
news_data = [
    {
        "id": "NEWS-001",
        "source": "Global Maritime Logistics",
        "text": "Naval patrols have been intensified around the Bab-el-Mandeb Strait following fresh drone sightings targeting container ships. Freight rates between East Asia and Europe rose by 18% overnight as insurers re-evaluate transit risks."
    },
    {
        "id": "NEWS-002",
        "source": "Cyber Threat Watch",
        "text": "A critical zero-day vulnerability (CVE-2026-8819) has been discovered in widespread enterprise VPN appliances. Threat actors are actively exploiting this unauthenticated remote code execution flaw to deploy ransomware."
    },
    {
        "id": "NEWS-003",
        "source": "Energy & Power Weekly",
        "text": "European natural gas storage levels reached 82% ahead of schedule following heavy LNG imports from Gulf suppliers. Regional pipeline operators report stable pressure despite seasonal heating surges."
    },
    {
        "id": "NEWS-004",
        "source": "Defense Tech Monitor",
        "text": "Joint NATO tactical exercises involving multi-domain unmanned ground vehicles (UGVs) kicked off in the Baltics today. The drill focuses on electronic warfare resilience and autonomous supply line logistics."
    }
]

with open("data/news_feed.json", "w", encoding="utf-8") as f:
    json.dump(news_data, f, indent=2, ensure_ascii=False)

print("✅ Data feed připraven v 'data/news_feed.json'")

In [ ]:
import os
import requests
import time
import pandas as pd
import urllib3
from IPython.display import display, JSON

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Model konfigurace — přímý KServe endpoint
MODEL_URL = os.environ.get("MODEL_URL",
    "https://qwen25-vl-7b-kserve-workload-svc.rhoai-playground.svc.cluster.local:8000/v1")
ENDPOINT_URL = MODEL_URL + "/chat/completions"
MODEL_NAME = "qwen25-vl-7b"

# Definice JSON Schema pro Guided Decoding
TRIAGE_JSON_SCHEMA = {
    "type": "object",
    "properties": {
        "category": {
            "type": "string",
            "enum": ["DEFENSE_SECURITY", "CYBERSECURITY", "ENERGY_COMMODITIES", "LOGISTICS_TRADE", "GENERAL_NEWS"]
        },
        "urgency_score": {
            "type": "integer",
            "minimum": 1,
            "maximum": 5
        },
        "summary": {
            "type": "string"
        },
        "key_entities": {
            "type": "array",
            "items": {"type": "string"}
        },
        "target_rag_index": {
            "type": "string",
            "enum": ["rag-defense-intel", "rag-cyber-threats", "rag-energy-market", "rag-general"]
        }
    },
    "required": ["category", "urgency_score", "summary", "key_entities", "target_rag_index"]
}

In [ ]:
def process_news_article(article_text, article_id):
    headers = {
        "Content-Type": "application/json"
    }
    
    system_prompt = (
        "You are an automated intelligence triage engine. "
        "Analyze the input news text and extract structured operational metadata according to the JSON schema."
    )
    
    payload = {
        "model": MODEL_NAME,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Article ID: {article_id}\nContent: {article_text}"}
        ],
        "temperature": 0.0,
        "max_tokens": 512,
        "response_format": {
            "type": "json_object",
            "schema": TRIAGE_JSON_SCHEMA
        }
    }
    
    start_time = time.time()
    
    response = requests.post(
        ENDPOINT_URL,
        headers=headers,
        json=payload,
        verify=False
    )
    
    elapsed_time = round(time.time() - start_time, 3)
    
    if response.status_code == 200:
        res_data = response.json()
        
        usage = res_data.get("usage", {})
        prompt_tokens = usage.get("prompt_tokens", 0)
        completion_tokens = usage.get("completion_tokens", 0)
        total_tokens = usage.get("total_tokens", 0)
        
        parsed_content = json.loads(res_data["choices"][0]["message"]["content"])
        
        return {
            "article_id": article_id,
            "latency_sec": elapsed_time,
            "prompt_tokens": prompt_tokens,
            "completion_tokens": completion_tokens,
            "total_tokens": total_tokens,
            "est_cost_usd": round((total_tokens / 1000) * 0.002, 6),
            "category": parsed_content.get("category"),
            "urgency": parsed_content.get("urgency_score"),
            "rag_target": parsed_content.get("target_rag_index"),
            "summary": parsed_content.get("summary"),
            "entities": ", ".join(parsed_content.get("key_entities", []))
        }
    else:
        print(f"❌ Error {response.status_code} on {article_id}: {response.text}")
        return None

In [ ]:
# Načtení dat a zpracování
with open("data/news_feed.json", "r", encoding="utf-8") as f:
    articles = json.load(f)

results = []
print("🚀 Spouštím dávkovou triáž přes OpenShift AI MaaS Gateway...\n")

for item in articles:
    res = process_news_article(item["text"], item["id"])
    if res:
        results.append(res)

df = pd.DataFrame(results)

print("=== MAAS METERING & TELEMETRY SUMMARY ===")
metrics_df = df[["article_id", "latency_sec", "prompt_tokens", "completion_tokens", "total_tokens", "est_cost_usd"]]
display(metrics_df)

print("\n=== STRUCTURED TRIAGE & RAG ROUTING RESULTS ===")
triage_df = df[["article_id", "category", "urgency", "rag_target", "entities"]]
display(triage_df)

In [ ]:
print("🔄 SIMULACE DYNAMICKÉHO SMĚROVÁNÍ DO RAG DATABÁZÍ:\n")

for idx, row in df.iterrows():
    print(f"📌 [Article {row['article_id']}] -> Route to Index: [{row['rag_target']}]")
    print(f"   Category: {row['category']} (Urgency: {row['urgency']}/5)")
    print(f"   Summary:  {row['summary']}")
    print(f"   Payload sent to pipeline: https://rag-ingest.internal.cluster/{row['rag_target']}/insert\n")